# Análise de Teste A/B de Cupom

## 1. Ingestão de dados

Baixamos e extraímos os dados brutos (pedidos, consumidores, restaurantes e referência do teste A/B) usando as funções reutilizáveis em `src/ingestion.py`. O arquivo de pedidos é grande (~3,6M de linhas), então usamos carregamento em streaming / em blocos em vez de ler tudo na memória de uma vez.


## 2. Limpeza de dados

Usando `src/processing.py`, padronizamos esquemas, convertemos timestamps, removemos pedidos duplicados, filtramos valores inválidos (por exemplo, valores de pedido não positivos) e tratamos dados faltantes. A saída final dessa etapa é a tabela `fact_orders` com as colunas requeridas e a flag de tratamento `is_target` mesclada a partir da referência A/B.


## 3. Engenharia de recursos

A partir de `fact_orders` calculamos métricas no estilo RFM por cliente em `src/feature_engineering.py`. Isso inclui total de pedidos, total gasto, valor médio do pedido, datas do primeiro/último pedido, recência (dias desde o último pedido), frequência e valor monetário. Esses recursos alimentam tanto a análise A/B (por usuário) quanto a etapa de segmentação.


## 4. Análise do teste A/B

Calculamos métricas A/B em `src/ab_test.py`: taxa de conversão, pedidos por usuário, GMV, GMV por usuário e valor médio do pedido, comparando alvo vs controle. Aplicamos um teste t de Welch em valores de pedido e um teste z de proporção nas taxas de conversão, reportando p-values e intervalos de confiança para avaliar significância estatística.


## 5. Segmentação

Usando `src/segmentation.py`, construímos segmentos baseados em RFM (Campeões, Fiéis, Potenciais, Em Risco, Baixo Engajamento). Em seguida, analisamos o desempenho do teste A/B por segmento para entender quais perfis de usuário respondem melhor à campanha de cupom e calculamos uplift em GMV por usuário para cada segmento.


## 6. Insights de negócio

Na seção final interpretamos as métricas e testes estatísticos para responder às principais questões de negócio:

- A campanha de cupom melhorou significativamente a conversão e/ou receita?
- A campanha é financeiramente viável após contabilizar o custo dos cupons e a taxa de repasse do iFood?
- Quais segmentos apresentam maior uplift e devem ser priorizados em campanhas futuras?

As tabelas correspondentes são salvas em `outputs/tables/` e gráficos (conversão, GMV por usuário, distribuição de valores de pedidos, uplift por segmento e distribuições RFM) são salvos em `outputs/charts/`.
